## Notebook to generate input files for Gaussian ESP calculations

**Created on 1st August, 2024**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_3d = True
import py3Dmol
from pymatgen.io.gaussian import Molecule, GaussianInput, GaussianOutput
from pymatgen.io.xyz import XYZ
import glob
import os, sys
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolDescriptors, PandasTools

In [3]:
%%bash
pwd
ls -ltr ../optimization/isolated-solv

/Users/riteshkumar/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/ESP-calcs
total 0
-rw-------@     1 riteshkumar  staff      325 Jul  4  2024 list_3rd_iter.csv
-rw-------@     1 riteshkumar  staff      270 Jul  4  2024 list_1st_iter.csv
drwx------  65535 riteshkumar  staff  2097120 Jul  5  2024 1st-iter
drwx------  65535 riteshkumar  staff  2097120 Jul  5  2024 3rd-iter
-rw-------@     1 riteshkumar  staff      198 Jul 11  2024 list_2nd_iter.csv
drwx------  65535 riteshkumar  staff  2097120 Jul 14  2024 2nd-iter
-rw-------@     1 riteshkumar  staff   113077 Aug  3  2024 gen_gaussian_input_smiles.ipynb
drwx------  65535 riteshkumar  staff  2097120 Aug  5  2024 lit-in-house
drwx------  65535 riteshkumar  staff  2097120 Aug  5  2024 sota_expt_omics
-rw-------@     1 riteshkumar  staff      351 Aug  5  2024 list_1st_iter_lumo.csv
-rw-------@     1 riteshkumar  staff      709 Aug 15  2024 list_comb.csv


### System: isolated solvents

#### For first iteration molecules

In [3]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/optimization/isolated-solv/1st-iter/') ## MBP path
log_list_1 = glob.glob("*.log")
log_list_1

['first_4.log',
 'first_5.log',
 'first_1.log',
 'first_0.log',
 'first_2.log',
 'first_3.log']

In [4]:
# done_list = [i if GaussianOutput(log[i]).properly_terminated == True else continue for in range(log_list)]
done_list_1 = [i for i in log_list_1 if GaussianOutput(i).properly_terminated == True]
done_list_1

['first_4.log',
 'first_5.log',
 'first_1.log',
 'first_0.log',
 'first_2.log',
 'first_3.log']

In [6]:
pym_mol_list_1 = [Molecule.from_file(i) for i in done_list_1]
pym_mol_list_1[0].num_sites

11

In [10]:
done_list_1[0].split('_')[1].split('.')[0]

'4'

In [ ]:
## From COMBAT database script:
# opt_gaussian_inputs = {
#     "functional": "B3LYP",
#     "basis_set": "6-31+G*",
#     "route_parameters": {
#         "Opt": "(calcfc, tight)",
#         "SCF": "Tight",
#         "int": "ultrafine",
#         "NoSymmetry": None,
#         "test": None,
#     },
#     "link0_parameters": {"%chk": "opt.chk", "%mem": "45GB", "%NProcShared": "28"},
# }

# freq_gaussian_inputs = {
#     "functional": "B3LYP",
#     "basis_set": "6-31+G*",
#     "route_parameters": {
#         "Freq": None,
#         "iop(7/33=1)": None,
#         "NoSymmetry": None,
#         "Polar": None,
#         "test": None,
#     },
#     "link0_parameters": {"%chk": "freq.chk", "%mem": "45GB", "%NProcShared": "28"},
# }

# esp_gaussian_inputs = {
#     "functional": "B3LYP",
#     "basis_set": "6-31+G*",
#     "route_parameters": {
#         "pop": "MK",
#         "iop(6/50=1)": None,
#         "NoSymmetry": None,
#         "Density": None,
#         "test": None,
#     },
#     "link0_parameters": {"%chk": "esp.chk", "%mem": "45GB", "%NProcShared": "28"},
# }

In [ ]:
## From chatgpt
# %chk=molecule_b3lyp_esp.chk
# #P B3LYP/6-31G(d) SCF=Tight Pop=MK IOp(6/33=2,6/41=10,6/42=17)

# Title Card: ESP calculation (B3LYP) for Molecule

# 0 1
# …

In [19]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/ESP-calcs/') ## change accordingly
for i in range(len(done_list_1)):
    # smile = df['smiles'][i]
    # mol = Chem.MolFromSmiles(smile)
    # num = rdMolDescriptors.CalcNumHeavyAtoms(mol)
    num = pym_mol_list_1[i].num_sites
    if num < 12:
        nprocs = 4; mem = 8
    elif num > 12 and num < 20:
        nprocs = 8; mem = 16
    elif num > 20 and num < 30:
        nprocs = 12; mem = 24
    elif num > 30 and num < 40:
        nprocs = 16; mem = 32
    elif num > 40 and num < 50:
        nprocs = 20; mem = 40
    else:
        nprocs = 24; mem = 48
    try:
        # pos, sym = gen_3d_pos(smile)
        # mol_pym = Molecule(sym, pos)
        mol_pym = pym_mol_list_1[i]
        title = 'scf_esp_' + done_list_1[i].split('_')[1].split('.')[0]
        input_file = 'isolated-solv/1st-iter/' + title + '.com'
        func = 'rb3lyp'                                                                 # name of functional used
        bas = '6-311++g(d,p)'                                                             # name of basis set used, not needed for semi-empirical calculations
        gau = GaussianInput(mol_pym,
        charge=0, spin_multiplicity=1,                                                 # uncomment it, if default values are not be used
        title=title, functional=func, basis_set=bas,                                      # remove only basis_set, if not necessary
        link0_parameters={"%nprocshared": nprocs, "%mem": str(mem)+'GB', "%chk": title + '.chk'},                         # nprocshared = required number of processors
        route_parameters={"scrf": "(pcm,solvent=thf)", "pop": "MK", "iop(6/50=1)": None, "NoSymmetry": None, 
                          "empiricaldispersion": "gd3bj"}, # edit this line very carefully
        dieze_tag = "#")                        # edit this line very carefully
        gau.write_file(input_file) 
    except ValueError:
        print('Manually generate for: ' + str(i))

In [18]:
os.getcwd()

'/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/ESP-calcs/isolated-solv/1st-iter'

#### For second iteration molecules

In [25]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/optimization/isolated-solv/2nd-iter/') ## MBP path
log_list_2 = glob.glob("*.log")
log_list_2

['second_3.log']

In [26]:
done_list_2 = [i for i in log_list_2 if GaussianOutput(i).properly_terminated == True]
done_list_2

['second_3.log']

In [27]:
pym_mol_list_2 = [Molecule.from_file(i) for i in done_list_2]
pym_mol_list_2[0].num_sites

21

In [28]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/ESP-calcs/') ## change accordingly
for i in range(len(done_list_2)):
    # smile = df['smiles'][i]
    # mol = Chem.MolFromSmiles(smile)
    # num = rdMolDescriptors.CalcNumHeavyAtoms(mol)
    num = pym_mol_list_2[i].num_sites
    if num < 12:
        nprocs = 4; mem = 8
    elif num > 12 and num < 20:
        nprocs = 8; mem = 16
    elif num > 20 and num < 30:
        nprocs = 12; mem = 24
    elif num > 30 and num < 40:
        nprocs = 16; mem = 32
    elif num > 40 and num < 50:
        nprocs = 20; mem = 40
    else:
        nprocs = 24; mem = 48
    try:
        # pos, sym = gen_3d_pos(smile)
        # mol_pym = Molecule(sym, pos)
        mol_pym = pym_mol_list_2[i]
        title = 'scf_esp_' + done_list_2[i].split('_')[1].split('.')[0]
        input_file = 'isolated-solv/2nd-iter/' + title + '.com'
        func = 'rb3lyp'                                                                 # name of functional used
        bas = '6-311++g(d,p)'                                                             # name of basis set used, not needed for semi-empirical calculations
        gau = GaussianInput(mol_pym,
        charge=0, spin_multiplicity=1,                                                 # uncomment it, if default values are not be used
        title=title, functional=func, basis_set=bas,                                      # remove only basis_set, if not necessary
        link0_parameters={"%nprocshared": nprocs, "%mem": str(mem)+'GB', "%chk": title + '.chk'},                         # nprocshared = required number of processors
        route_parameters={"scrf": "(pcm,solvent=thf)", "pop": "MK", "iop(6/50=1)": None, "NoSymmetry": None}, # edit this line very carefully
        dieze_tag = "#")                        # edit this line very carefully
        gau.write_file(input_file) 
    except ValueError:
        print('Manually generate for: ' + str(i))

#### For third iteration molecules

In [31]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/optimization/isolated-solv/3rd-iter/') ## MBP path
log_list_3 = glob.glob("*.log")
log_list_3

['third_8.log',
 'third_3.log',
 'third_2.log',
 'third_0.log',
 'third_1.log',
 'third_5.log',
 'third_4.log',
 'third_6.log',
 'third_7.log']

In [22]:
done_list_3 = [i for i in log_list_3 if GaussianOutput(i).properly_terminated == True]
done_list_3

['third_8.log',
 'third_3.log',
 'third_2.log',
 'third_0.log',
 'third_1.log',
 'third_5.log',
 'third_4.log',
 'third_6.log',
 'third_7.log']

In [32]:
pym_mol_list_3 = [Molecule.from_file(i) for i in done_list_3]
pym_mol_list_3[0].num_sites

22

In [33]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/ESP-calcs/') ## change accordingly
for i in range(len(done_list_3)):
    # smile = df['smiles'][i]
    # mol = Chem.MolFromSmiles(smile)
    # num = rdMolDescriptors.CalcNumHeavyAtoms(mol)
    num = pym_mol_list_3[i].num_sites
    if num < 12:
        nprocs = 4; mem = 8
    elif num > 12 and num < 20:
        nprocs = 8; mem = 16
    elif num > 20 and num < 30:
        nprocs = 12; mem = 24
    elif num > 30 and num < 40:
        nprocs = 16; mem = 32
    elif num > 40 and num < 50:
        nprocs = 20; mem = 40
    else:
        nprocs = 24; mem = 48
    try:
        # pos, sym = gen_3d_pos(smile)
        # mol_pym = Molecule(sym, pos)
        mol_pym = pym_mol_list_3[i]
        title = 'scf_esp_' + done_list_3[i].split('_')[1].split('.')[0]
        input_file = 'isolated-solv/3rd-iter/' + title + '.com'
        func = 'rb3lyp'                                                                 # name of functional used
        bas = '6-311++g(d,p)'                                                             # name of basis set used, not needed for semi-empirical calculations
        gau = GaussianInput(mol_pym,
        charge=0, spin_multiplicity=1,                                                 # uncomment it, if default values are not be used
        title=title, functional=func, basis_set=bas,                                      # remove only basis_set, if not necessary
        link0_parameters={"%nprocshared": nprocs, "%mem": str(mem)+'GB', "%chk": title + '.chk'},                         # nprocshared = required number of processors
        route_parameters={"scrf": "(pcm,solvent=thf)", "pop": "MK", "iop(6/50=1)": None, "NoSymmetry": None}, # edit this line very carefully; GD3BJ does not work in Gaussian 09 with pop options
        dieze_tag = "#")                        # edit this line very carefully
        gau.write_file(input_file) 
    except ValueError:
        print('Manually generate for: ' + str(i))

#### For OOD molecules

In [15]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/optimization/isolated-solv/lit-in-house/') ## MBP path
log_list_1 = glob.glob("*.log")
log_list_1

['ood_2.log',
 'ood_3.log',
 'ood_1.log',
 'ood_0.log',
 'ood_4.log',
 'ood_5.log',
 'ood_7.log',
 'ood_6.log',
 'ood_17.log',
 'ood_16.log',
 'ood_14.log',
 'ood_15.log',
 'ood_11.log',
 'ood_10.log',
 'ood_22.log',
 'ood_23.log',
 'ood_20.log',
 'ood_24.log',
 'ood_18.log',
 'ood_19.log',
 'ood_25.log',
 'ood_8.log',
 'ood_9.log']

In [16]:
done_list_1 = [i for i in log_list_1 if GaussianOutput(i).properly_terminated == True]
len(done_list_1)

23

In [17]:
pym_mol_list_1 = [Molecule.from_file(i) for i in done_list_1]
pym_mol_list_1[0].num_sites

26

In [18]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/ESP-calcs/') ## change accordingly
for i in range(len(done_list_1)):
    # smile = df['smiles'][i]
    # mol = Chem.MolFromSmiles(smile)
    # num = rdMolDescriptors.CalcNumHeavyAtoms(mol)
    num = pym_mol_list_1[i].num_sites
    if num < 12:
        nprocs = 4; mem = 8
    elif num > 12 and num < 20:
        nprocs = 8; mem = 16
    elif num > 20 and num < 30:
        nprocs = 12; mem = 24
    elif num > 30 and num < 40:
        nprocs = 16; mem = 32
    elif num > 40 and num < 50:
        nprocs = 20; mem = 40
    else:
        nprocs = 24; mem = 48
    try:
        # pos, sym = gen_3d_pos(smile)
        # mol_pym = Molecule(sym, pos)
        mol_pym = pym_mol_list_1[i]
        title = 'scf_esp_' + done_list_1[i].split('_')[1].split('.')[0]
        input_file = 'isolated-solv/lit-in-house/' + title + '.com'
        func = 'rb3lyp'                                                                 # name of functional used
        bas = '6-311++g(d,p)'                                                             # name of basis set used, not needed for semi-empirical calculations
        gau = GaussianInput(mol_pym,
        charge=0, spin_multiplicity=1,                                                 # uncomment it, if default values are not be used
        title=title, functional=func, basis_set=bas,                                      # remove only basis_set, if not necessary
        link0_parameters={"%nprocshared": nprocs, "%mem": str(mem)+'GB', "%chk": title + '.chk'},                         # nprocshared = required number of processors
        route_parameters={"scrf": "(pcm,solvent=thf)", "pop": "MK", "iop(6/50=1)": None, "NoSymmetry": None}, 
                        #   "empiricaldispersion": "gd3bj"}, # edit this line very carefully; empirical dispersion does not work for ESP??
        dieze_tag = "#")                        # edit this line very carefully
        gau.write_file(input_file) 
    except ValueError:
        print('Manually generate for: ' + str(i))

#### For SOTA molecules

In [9]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/optimization/isolated-solv/sota_expt_omics/') ## MBP path
log_list_1 = glob.glob("*.log")
log_list_1

['sota_2.log',
 'sota_0.log',
 'sota_5.log',
 'sota_4.log',
 'sota_7.log',
 'sota_10.log',
 'sota_9.log',
 'sota_8.log']

In [10]:
done_list_1 = [i for i in log_list_1 if GaussianOutput(i).properly_terminated == True]
(len(done_list_1))

8

In [11]:
pym_mol_list_1 = [Molecule.from_file(i) for i in done_list_1]
pym_mol_list_1[0].num_sites

18

In [14]:
os.chdir('/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/ESP-calcs/') ## change accordingly
for i in range(len(done_list_1)):
    # smile = df['smiles'][i]
    # mol = Chem.MolFromSmiles(smile)
    # num = rdMolDescriptors.CalcNumHeavyAtoms(mol)
    num = pym_mol_list_1[i].num_sites
    if num < 12:
        nprocs = 4; mem = 8
    elif num > 12 and num < 20:
        nprocs = 8; mem = 16
    elif num > 20 and num < 30:
        nprocs = 12; mem = 24
    elif num > 30 and num < 40:
        nprocs = 16; mem = 32
    elif num > 40 and num < 50:
        nprocs = 20; mem = 40
    else:
        nprocs = 24; mem = 48
    try:
        # pos, sym = gen_3d_pos(smile)
        # mol_pym = Molecule(sym, pos)
        mol_pym = pym_mol_list_1[i]
        title = 'scf_esp_' + done_list_1[i].split('_')[1].split('.')[0]
        input_file = 'isolated-solv/sota_expt_omics/' + title + '.com'
        func = 'rb3lyp'                                                                 # name of functional used
        bas = '6-311++g(d,p)'                                                             # name of basis set used, not needed for semi-empirical calculations
        gau = GaussianInput(mol_pym,
        charge=0, spin_multiplicity=1,                                                 # uncomment it, if default values are not be used
        title=title, functional=func, basis_set=bas,                                      # remove only basis_set, if not necessary
        link0_parameters={"%nprocshared": nprocs, "%mem": str(mem)+'GB', "%chk": title + '.chk'},                         # nprocshared = required number of processors
        route_parameters={"scrf": "(pcm,solvent=thf)", "pop": "MK", "iop(6/50=1)": None, "NoSymmetry": None},  ## does not work with ESP calculations??
                        #   "empiricaldispersion": "gd3bj"}, # edit this line very carefully
        dieze_tag = "#")                        # edit this line very carefully
        gau.write_file(input_file) 
    except ValueError:
        print('Manually generate for: ' + str(i))

#### For FSI- molecule

In [5]:
os.chdir('/Users/riteshkumar/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/optimization/FSA-/') ## MBP path
log_list_1 = glob.glob("*.log")
log_list_1

['fsa-.log']

In [6]:
done_list_1 = [i for i in log_list_1 if GaussianOutput(i).properly_terminated == True]
(len(done_list_1))

1

In [7]:
pym_mol_list_1 = [Molecule.from_file(i) for i in done_list_1]
pym_mol_list_1[0].num_sites

9

In [13]:
os.chdir('/Users/riteshkumar/Library/CloudStorage/Box-Box/Research-postdoc/eScore-extension/gaussian-dft-calcs/ESP-calcs/') ## change accordingly
for i in range(len(done_list_1)):
    # smile = df['smiles'][i]
    # mol = Chem.MolFromSmiles(smile)
    # num = rdMolDescriptors.CalcNumHeavyAtoms(mol)
    num = pym_mol_list_1[i].num_sites
    if num < 12:
        nprocs = 4; mem = 8
    elif num > 12 and num < 20:
        nprocs = 8; mem = 16
    elif num > 20 and num < 30:
        nprocs = 12; mem = 24
    elif num > 30 and num < 40:
        nprocs = 16; mem = 32
    elif num > 40 and num < 50:
        nprocs = 20; mem = 40
    else:
        nprocs = 24; mem = 48
    try:
        # pos, sym = gen_3d_pos(smile)
        # mol_pym = Molecule(sym, pos)
        mol_pym = pym_mol_list_1[i]
        print(mol_pym)
        title = 'scf_esp_' + done_list_1[i].split('.')[0]
        # title = 'scf_esp_' + done_list_1[i].split('_')[1].split('.')[0]
        input_file = 'FSA-/' + title + '.com' ## change accordingly
        func = 'rb3lyp'                                                                 # name of functional used
        bas = '6-311++g(d,p)'                                                             # name of basis set used, not needed for semi-empirical calculations
        gau = GaussianInput(mol_pym,
        charge=-1, spin_multiplicity=1,                                                 # uncomment it, if default values are not be used
        title=title, functional=func, basis_set=bas,                                      # remove only basis_set, if not necessary
        link0_parameters={"%nprocshared": nprocs, "%mem": str(mem)+'GB', "%chk": title + '.chk'},                         # nprocshared = required number of processors
        route_parameters={"scrf": "(pcm,solvent=thf)", "pop": "MK", "iop(6/50=1)": None, "NoSymmetry": None},  ## does not work with ESP calculations??
                        #   "empiricaldispersion": "gd3bj"}, # edit this line very carefully
        dieze_tag = "#")                        # edit this line very carefully
        gau.write_file(input_file) 
    except ValueError:
        print('Manually generate for: ' + str(i))

Full Formula (S2 N1 O4 F2)
Reduced Formula: S2N(O2F)2
Charge = 0.0, Spin Mult = 2
Sites (9)
0 F     1.690311    -1.398141    -0.424086
1 F    -1.690342     1.398195    -0.423839
2 S     1.391147     0.156370     0.089160
3 S    -1.391143    -0.156381     0.089181
4 O     2.433626     0.400381     1.064322
5 O     1.358502     0.919570    -1.145752
6 O    -2.433590    -0.400539     1.064340
7 O    -1.358524    -0.919406    -1.145839
8 N     0.000014    -0.000053     0.868756
